# Notebook 4 — `inference.ipynb`

**Sovereign Dialect-Bridge · Step 4 — Evaluation + End-to-End Demo**

Evaluasi dua level:

| Level | Apa | Metrik |
|-------|-----|--------|
| **L1 per-stage** | Normalizer pada NusaX val | BLEU-4, chrF++ |
| **L1 per-stage** | Summarizer + baseline pada IndoSum test (700 stratified) | ROUGE-1/2/L, BERTScore-F1, CR |
| **L2 end-to-end** | Dialect test set (NusaX pseudo-articles, 3 dialek) | ROUGE without_norm vs with_norm, **Δ per dialek** |

Output:
- `outputs/final_results.json` — semua metrik + prediksi + metadata
- Tabel hasil + 3-5 contoh konkret per metode
- Demo `DialectBridge.bridge()` untuk teks dialek arbitrer


## 1. Setup environment


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]   = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc, json, time, sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = torch.cuda.is_bf16_supported() if DEVICE == "cuda" else False

print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}  ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
print(f"bf16   : {USE_BF16}")


## 2. Install dependencies

```bash
pip install transformers==4.40.0 datasets accelerate sentencepiece
pip install rouge-score bert-score sacrebleu sacremoses
pip install PySastrawi networkx scikit-learn
```


## 3. Paths + import helpers from Notebook 3


In [ ]:
CWD = Path.cwd()
ROOT = CWD if (CWD / "data").exists() else CWD.parent
DATA_DIR    = ROOT / "data"
MODELS_DIR  = ROOT / "models"
OUTPUTS_DIR = ROOT / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Import shared helpers (saved by notebook 3)
sys.path.insert(0, str(ROOT / "notebook"))
from _helpers import (
    preprocess_extractive, preprocess_abstractive,
    summarize_textrank, summarize_ner, split_sentences, GEN_KWARGS,
)

NORM_DIR     = MODELS_DIR / "normalizer"
INDOT5_DIR   = MODELS_DIR / "indot5"
INDOBART_DIR = MODELS_DIR / "indobart"
MT5BASE_DIR  = MODELS_DIR / "mt5base"

for p in [NORM_DIR, INDOT5_DIR, INDOBART_DIR, MT5BASE_DIR]:
    print(f"  {p.name:12s} : {'✓' if p.exists() else '✗ MISSING'}")


## 4. Load test data


In [ ]:
df_test = pd.read_parquet(DATA_DIR / "test.parquet")
print(f"Test: {len(df_test):,}")
print(f"Categories: {df_test['category'].value_counts().to_dict()}")


## 5. Evaluation metric helpers


In [ ]:
from rouge_score import rouge_scorer as rs
from bert_score import score as bert_score_fn


def compute_rouge(preds, refs):
    scorer = rs.RougeScorer(["rouge1","rouge2","rougeL"], use_stemmer=False)
    scores = [scorer.score(r, p) for p, r in zip(preds, refs)]
    n = len(scores)
    return {
        "rouge1": round(sum(s["rouge1"].fmeasure for s in scores) / n, 4),
        "rouge2": round(sum(s["rouge2"].fmeasure for s in scores) / n, 4),
        "rougeL": round(sum(s["rougeL"].fmeasure for s in scores) / n, 4),
    }


def compute_bertscore(preds, refs, lang="id"):
    if not preds:
        return {"bertscore_p": None, "bertscore_r": None, "bertscore_f1": None}
    P, R, F1 = bert_score_fn(preds, refs, lang=lang, verbose=False, device=DEVICE)
    return {
        "bertscore_p":  round(P.mean().item(),  4),
        "bertscore_r":  round(R.mean().item(),  4),
        "bertscore_f1": round(F1.mean().item(), 4),
    }


def compute_cr(preds, originals):
    crs = [len(p.split()) / max(len(o.split()), 1) for p, o in zip(preds, originals)]
    return round(sum(crs) / len(crs), 4)


## 6. Summarizer inference helper


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


def free_vram(*objs):
    for o in objs:
        del o
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def generate_summaries(model_dir, model_type, df, max_input=512):
    """Load checkpoint, generate untuk semua row di df, return list of predictions."""
    print(f"  Loading {model_dir} ...")
    tokenizer = AutoTokenizer.from_pretrained(str(model_dir),
                                              use_fast=False if model_type == "bart" else True)
    model = AutoModelForSeq2SeqLM.from_pretrained(str(model_dir)).to(DEVICE)
    model.eval()

    preds = []
    t0 = time.time()
    for i, text in enumerate(df["text"].tolist()):
        text = preprocess_abstractive(text)
        if model_type == "mt5":
            text = "summarize: " + text
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_input).to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inputs, **GEN_KWARGS)
        preds.append(tokenizer.decode(out[0], skip_special_tokens=True))
        if (i+1) % 50 == 0:
            print(f"    {i+1}/{len(df)}  elapsed={time.time()-t0:.0f}s")

    free_vram(model, tokenizer)
    return preds


## 7. Generate prediksi summarizer (3 model) + baseline (TextRank, NER)


In [ ]:
all_preds = {}

print("\n--- TextRank ---")
all_preds["TextRank"] = [summarize_textrank(t, n_sentences=3, max_words=80) for t in df_test["text"]]
print(f"  done: {len(all_preds['TextRank'])} preds")

print("\n--- NER ---")
all_preds["NER"] = [summarize_ner(t, n_sentences=3, max_words=80) for t in df_test["text"]]
print(f"  done: {len(all_preds['NER'])} preds")

print("\n--- IndoT5 ---")
all_preds["IndoT5"] = generate_summaries(INDOT5_DIR, "t5", df_test)

print("\n--- IndoBART ---")
all_preds["IndoBART"] = generate_summaries(INDOBART_DIR, "bart", df_test)

print("\n--- mT5-base ---")
all_preds["mT5-base"] = generate_summaries(MT5BASE_DIR, "mt5", df_test)

print(f"\nTotal methods evaluated: {len(all_preds)}")


## 8. Hitung metrik Level 1 — tabel utama


In [ ]:
refs      = df_test["summary"].astype(str).tolist()
originals = df_test["text"].astype(str).tolist()

results_rows = []
for method, preds in all_preds.items():
    print(f"\nComputing metrics for {method} ...")
    r = compute_rouge(preds, refs)
    cr = compute_cr(preds, originals)
    row = {
        "method": method,
        "rouge1": r["rouge1"], "rouge2": r["rouge2"], "rougeL": r["rougeL"],
        "cr": cr, "n": len(preds),
    }
    if method in {"IndoT5", "IndoBART", "mT5-base"}:
        b = compute_bertscore(preds, refs, lang="id")
        row["bertscore_f1"] = b["bertscore_f1"]
    else:
        row["bertscore_f1"] = None
    results_rows.append(row)
    print(f"  R1={row['rouge1']:.4f}  R2={row['rouge2']:.4f}  RL={row['rougeL']:.4f}  CR={row['cr']:.4f}")

results_df = pd.DataFrame(results_rows)
print("\n" + "="*70)
print("Level 1 — Per-stage Summarizer Evaluation")
print("="*70)
print(results_df.to_string(index=False))


## 9. Evaluasi Normalizer — BLEU-4 + chrF++


In [ ]:
import sacrebleu
from datasets import load_dataset

# Load NusaX val (lokal preferred)
nusax_val = None
local_path = ROOT / "dataset" / "nusax" / "datasets" / "mt" / "valid.csv"
if local_path.exists():
    nusax_val = pd.read_csv(local_path)
    print(f"NusaX val (local): {len(nusax_val):,} rows, cols={list(nusax_val.columns)}")
else:
    try:
        ds = load_dataset("indonlp/NusaX-MT", trust_remote_code=True)
        nusax_val = ds["validation"].to_pandas() if "validation" in ds else ds["valid"].to_pandas()
    except Exception as e:
        print(f"Failed to load NusaX val: {e}")

if nusax_val is not None and NORM_DIR.exists():
    norm_tokenizer = AutoTokenizer.from_pretrained(str(NORM_DIR))
    norm_model     = AutoModelForSeq2SeqLM.from_pretrained(str(NORM_DIR)).to(DEVICE)
    norm_model.eval()

    DIALECT_COLS = [c for c in nusax_val.columns
                    if c not in ("indonesian", "english") and not c.startswith("Unnamed")]
    print(f"Eval dialect columns: {DIALECT_COLS}")

    preds_n, refs_n = [], []
    SAMPLE_PER_DIALECT = min(20, len(nusax_val))
    for d in DIALECT_COLS:
        sample = nusax_val[["indonesian", d]].dropna().head(SAMPLE_PER_DIALECT)
        for _, row in sample.iterrows():
            inputs = norm_tokenizer(str(row[d]), return_tensors="pt",
                                    truncation=True, max_length=256).to(DEVICE)
            with torch.no_grad():
                out = norm_model.generate(**inputs, max_new_tokens=256, num_beams=4,
                                          no_repeat_ngram_size=3, early_stopping=True)
            preds_n.append(norm_tokenizer.decode(out[0], skip_special_tokens=True))
            refs_n.append(str(row["indonesian"]))

    bleu = sacrebleu.corpus_bleu(preds_n, [refs_n])
    chrf = sacrebleu.corpus_chrf(preds_n, [refs_n])
    norm_metrics = {"bleu4": round(bleu.score, 2), "chrf": round(chrf.score, 2),
                    "n_samples": len(preds_n)}
    print(f"\nNormalizer  BLEU-4 = {norm_metrics['bleu4']:.2f}  (target: > 20)")
    print(f"            chrF++ = {norm_metrics['chrf']:.2f}")
    free_vram(norm_model, norm_tokenizer)
else:
    norm_metrics = {"bleu4": None, "chrf": None, "n_samples": 0}
    print("Skipped: NusaX val or normalizer model not found.")


## 10. DialectBridge class — end-to-end pipeline


In [ ]:
class DialectBridge:
    """
    Two-stage dialect summarization pipeline.

    Stage 1 (Normalizer): dialect text → Bahasa Indonesia baku  (mT5-small)
    Stage 2 (Summarizer): BI text     → BI summary              (IndoT5 / IndoBART / mT5-base)
    """

    SUPPORTED_SUMMARIZERS = {
        "indot5":   (str(INDOT5_DIR),   "t5"),
        "indobart": (str(INDOBART_DIR), "bart"),
        "mt5base":  (str(MT5BASE_DIR),  "mt5"),
    }

    def __init__(self, summarizer="indot5"):
        if summarizer not in self.SUPPORTED_SUMMARIZERS:
            raise ValueError(f"Unsupported summarizer: {summarizer}")
        self.summarizer_name = summarizer

        self.norm_tok = AutoTokenizer.from_pretrained(str(NORM_DIR))
        self.norm_mod = AutoModelForSeq2SeqLM.from_pretrained(str(NORM_DIR)).to(DEVICE).eval()

        sum_path, sum_type = self.SUPPORTED_SUMMARIZERS[summarizer]
        self.sum_tok  = AutoTokenizer.from_pretrained(sum_path,
                                                     use_fast=False if sum_type=="bart" else True)
        self.sum_mod  = AutoModelForSeq2SeqLM.from_pretrained(sum_path).to(DEVICE).eval()
        self.sum_type = sum_type

    def normalize(self, text: str) -> str:
        inputs = self.norm_tok(text, return_tensors="pt", truncation=True, max_length=256).to(DEVICE)
        with torch.no_grad():
            out = self.norm_mod.generate(**inputs, max_new_tokens=256, num_beams=4,
                                         no_repeat_ngram_size=3, early_stopping=True)
        return self.norm_tok.decode(out[0], skip_special_tokens=True)

    def summarize(self, text: str) -> str:
        t = preprocess_abstractive(text)
        if self.sum_type == "mt5":
            t = "summarize: " + t
        inputs = self.sum_tok(t, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
        with torch.no_grad():
            out = self.sum_mod.generate(**inputs, **GEN_KWARGS)
        return self.sum_tok.decode(out[0], skip_special_tokens=True)

    def bridge(self, dialect_text: str, verbose: bool = True) -> dict:
        """Full pipeline: dialect text → BI formal summary."""
        t0 = time.time()
        normalized = self.normalize(dialect_text)

        # QC gate: output normalisasi harus > 30% panjang input
        n_in  = len(dialect_text.split())
        n_out = len(normalized.split())
        qc_passed = (n_out / max(n_in, 1)) >= 0.30 if n_in > 0 else False

        summary = self.summarize(normalized if qc_passed else dialect_text)
        elapsed = round(time.time() - t0, 3)

        result = {
            "input_dialect"    : dialect_text,
            "normalized_bi"    : normalized,
            "qc_passed"        : bool(qc_passed),
            "summary"          : summary,
            "summarizer_used"  : self.summarizer_name,
            "compression_ratio": round(len(summary.split()) / max(n_in, 1), 4),
            "elapsed_sec"      : elapsed,
        }
        if verbose:
            print(f"[DialectBridge:{self.summarizer_name}]  elapsed={elapsed}s  qc={qc_passed}")
            print(f"  INPUT      : {dialect_text[:100]}")
            print(f"  NORMALIZED : {normalized[:100]}")
            print(f"  SUMMARY    : {summary[:100]}")
        return result

    def free(self):
        free_vram(self.norm_mod, self.sum_mod, self.norm_tok, self.sum_tok)


## 11. Build dialect test set + Level 2 evaluation


In [ ]:
def build_dialect_test_set(df_nusax, dialect_col, n_articles=30, n_sents=5):
    """Gabung kalimat NusaX jadi pseudo-artikel (eval-only, BUKAN training)."""
    rows = df_nusax[["indonesian", dialect_col]].dropna().reset_index(drop=True)
    rows = rows[rows[dialect_col].astype(str).str.strip().astype(bool)]
    articles = []
    for start in range(0, min(n_articles * n_sents, len(rows)), n_sents):
        chunk = rows.iloc[start:start + n_sents]
        if len(chunk) < n_sents:
            break
        articles.append({
            "text_dialect"   : " ".join(str(x) for x in chunk[dialect_col].tolist()),
            "text_indonesian": " ".join(str(x) for x in chunk["indonesian"].tolist()),
            "summary"        : str(rows.iloc[start]["indonesian"]),
            "dialect"        : dialect_col,
        })
    return articles


# Build dialect test set from NusaX (combine train + valid + test for max coverage)
nusax_full = None
local_dir = ROOT / "dataset" / "nusax" / "datasets" / "mt"
if local_dir.exists():
    nusax_full = pd.concat(
        [pd.read_csv(local_dir / f) for f in ["train.csv", "valid.csv", "test.csv"] if (local_dir / f).exists()],
        ignore_index=True,
    )

dialect_test = []
if nusax_full is not None:
    for d in ["javanese", "sundanese", "minangkabau"]:
        if d in nusax_full.columns:
            dialect_test.extend(build_dialect_test_set(nusax_full, d, n_articles=30, n_sents=5))

print(f"Dialect test set: {len(dialect_test)} pseudo-articles")
if dialect_test:
    print(f"  Sample dialect text: {dialect_test[0]['text_dialect'][:120]}")


In [ ]:
def eval_dialect_robustness(summarizer_key, dialect_articles):
    """Compare ROUGE: raw dialect → summarizer  vs.  dialect → normalizer → summarizer."""
    if not dialect_articles:
        return {"without_norm": None, "with_norm": None, "delta": None, "per_dialect": {}}

    bridge = DialectBridge(summarizer=summarizer_key)

    scorer = rs.RougeScorer(["rouge1","rouge2","rougeL"], use_stemmer=False)

    def agg_rouge(preds, refs):
        scores = [scorer.score(r, p) for p, r in zip(preds, refs)]
        n = len(scores)
        return {k: round(sum(s[k].fmeasure for s in scores) / n, 4)
                for k in ("rouge1","rouge2","rougeL")}

    refs_all       = [a["summary"] for a in dialect_articles]
    preds_without  = []
    preds_with     = []
    per_dialect_rows = []

    for art in dialect_articles:
        # Without normalization: pakai raw dialect text langsung ke summarizer
        preds_without.append(bridge.summarize(art["text_dialect"]))
        # With normalization: full bridge
        res = bridge.bridge(art["text_dialect"], verbose=False)
        preds_with.append(res["summary"])

    overall_without = agg_rouge(preds_without, refs_all)
    overall_with    = agg_rouge(preds_with,    refs_all)

    # Per dialect breakdown
    dialects = sorted(set(a["dialect"] for a in dialect_articles))
    per_dialect = {}
    for d in dialects:
        idx = [i for i, a in enumerate(dialect_articles) if a["dialect"] == d]
        w = agg_rouge([preds_without[i] for i in idx], [refs_all[i] for i in idx])
        n = agg_rouge([preds_with[i]    for i in idx], [refs_all[i] for i in idx])
        per_dialect[d] = {"without_norm": w, "with_norm": n,
                          "delta_rouge1": round(n["rouge1"] - w["rouge1"], 4)}

    bridge.free()
    return {
        "without_norm": overall_without,
        "with_norm"   : overall_with,
        "delta_rouge1": round(overall_with["rouge1"] - overall_without["rouge1"], 4),
        "per_dialect" : per_dialect,
        "preds_without": preds_without,
        "preds_with"   : preds_with,
    }


robustness = {}
if dialect_test and NORM_DIR.exists():
    for summ_key in ["indot5", "indobart"]:
        summ_path = MODELS_DIR / (summ_key if summ_key != "mt5base" else "mt5base")
        if summ_path.exists():
            print(f"\n=== Robustness eval: {summ_key} ===")
            robustness[summ_key] = eval_dialect_robustness(summ_key, dialect_test)
            r = robustness[summ_key]
            print(f"  Without norm: {r['without_norm']}")
            print(f"  With    norm: {r['with_norm']}")
            print(f"  Δ ROUGE-1   : {r['delta_rouge1']:+.4f}")
else:
    print("Skipped robustness eval — dialect test set or normalizer missing.")


## 12. Tabel hasil final


In [ ]:
# Format tabel utama
print("=" * 80)
print("Final results — Level 1 (Summarizer evaluation on IndoSum test, n=" + str(len(df_test)) + ")")
print("=" * 80)
print(f"{'Method':<12}{'ROUGE1':>10}{'ROUGE2':>10}{'ROUGEL':>10}{'BERT-F1':>12}{'CR':>8}")
print("-" * 80)
for row in results_rows:
    bs = f"{row['bertscore_f1']:.4f}" if row['bertscore_f1'] is not None else "—"
    print(f"{row['method']:<12}{row['rouge1']:>10.4f}{row['rouge2']:>10.4f}"
          f"{row['rougeL']:>10.4f}{bs:>12}{row['cr']:>8.4f}")

print()
print("=" * 80)
print("Normalizer (Stage 1) — NusaX validation")
print("=" * 80)
print(f"  BLEU-4 : {norm_metrics.get('bleu4')}  (target > 20)")
print(f"  chrF++ : {norm_metrics.get('chrf')}")
print(f"  n      : {norm_metrics.get('n_samples')}")

if robustness:
    print()
    print("=" * 80)
    print("Level 2 — Dialect Robustness (NusaX pseudo-articles)")
    print("=" * 80)
    print(f"{'Summarizer':<14}{'Without Norm R1':>18}{'With Norm R1':>18}{'Δ R1':>10}")
    print("-" * 80)
    for k, r in robustness.items():
        wn = r['without_norm']['rouge1'] if r.get('without_norm') else None
        wi = r['with_norm']['rouge1']    if r.get('with_norm') else None
        d  = r.get('delta_rouge1')
        wn_s = f"{wn:.4f}" if wn is not None else "—"
        wi_s = f"{wi:.4f}" if wi is not None else "—"
        d_s  = f"{d:+.4f}" if d is not None else "—"
        print(f"{k:<14}{wn_s:>18}{wi_s:>18}{d_s:>10}")
    print("\n  Δ > 0  =>  bukti empiris Dialect Bridge bekerja ✓")


## 13. Contoh konkret per metode (3 samples)


In [ ]:
sample_idx = list(range(min(3, len(df_test))))

for idx in sample_idx:
    art  = df_test.iloc[idx]
    print("=" * 80)
    print(f"SAMPLE #{idx}  |  category={art['category']}")
    print("=" * 80)
    print(f"ARTICLE ({art['word_count']} words):")
    print(f"  {art['text'][:280]}...")
    print()
    print(f"REFERENCE SUMMARY ({art['summary_word_count']} words):")
    print(f"  {art['summary'][:200]}")
    print()
    for method in ["TextRank", "NER", "IndoT5", "IndoBART", "mT5-base"]:
        pred = all_preds[method][idx]
        n_w = len(pred.split())
        print(f"[{method}] ({n_w} words):")
        print(f"  {pred[:200]}")
    print()


## 14. End-to-end demo — DialectBridge


In [ ]:
DEMO_INPUTS = [
    ("Kulo lapor dalan teng ngajeng griyo kulo rusak parah, sampun dangu mboten dipun beton. "
     "Warga ingkang langkung saking margi punika sring kacilakan motoripun.", "javanese"),
    ("Abdi ngalaporkeun yén jalan di payun bumi abdi parah pisan, atos lami teu diaspal. "
     "Warga anu ngalangkungan jalan éta sering kacilakaan motorna.", "sundanese"),
    ("Ambo malaporan jalan di muko rumah ambo rusak bana, alah lamo indak dibeton. "
     "Urang nan lewat di jalan ko sering kacilakaan motonyo.", "minangkabau"),
]

if NORM_DIR.exists() and INDOT5_DIR.exists():
    bridge_demo = DialectBridge(summarizer="indot5")
    demo_outputs = []
    for text, dialect in DEMO_INPUTS:
        print(f"\n──── {dialect.upper()} ────")
        out = bridge_demo.bridge(text, verbose=True)
        out["dialect"] = dialect
        demo_outputs.append(out)
    bridge_demo.free()
else:
    demo_outputs = []
    print("Skipped demo — model files missing.")


## 15. Save final results


In [ ]:
final = {
    "metadata": {
        "n_test_samples": len(df_test),
        "models": {
            "normalizer": NORM_MODEL if False else "google/mt5-small",
            "indot5": "cahya/t5-base-indonesian-summarization-cased",
            "indobart": "indobenchmark/indoBART",
            "mt5base": "google/mt5-base",
        },
    },
    "level1": {
        "summarizers": results_rows,
        "normalizer" : norm_metrics,
    },
    "level2_robustness": {
        k: {kk: vv for kk, vv in v.items() if kk != "preds_without" and kk != "preds_with"}
        for k, v in robustness.items()
    } if robustness else {},
    "demo_outputs": demo_outputs,
    "examples": [
        {
            "idx": i,
            "category": df_test.iloc[i]["category"],
            "reference_summary": df_test.iloc[i]["summary"],
            "predictions": {m: all_preds[m][i] for m in all_preds},
        }
        for i in sample_idx
    ],
}

out_path = OUTPUTS_DIR / "final_results.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(final, f, ensure_ascii=False, indent=2)
print(f"\n✓ Saved → {out_path}")
print(f"  size = {out_path.stat().st_size / 1024:.1f} KB")


## ✅ Selesai

Output:
- `outputs/final_results.json` — semua metrik + prediksi + demo

**Bagaimana membaca hasilnya:**
- **L1 Summarizer:** target ROUGE-1 > 0.30 untuk model abstractive, CR ~0.22 (natural IndoSum)
- **Normalizer:** target BLEU-4 > 20 di NusaX validation
- **L2 Robustness:** Δ positif pada at-least 1 dialek = bukti empiris pipeline Two-Stage bekerja

Selesai. 🎉
